# Phase 4a: Preprocess Steering Vector Training Data

**Goal**: build per-cluster training examples `(prompt, prefix_sentences, target_sentence, sae_activation)` for steering-vector training.

Per paper (Appendix C.1): for each cluster c, find sentences where cluster c's latent fires strongest (top-2048 by SAE activation). Paper also filters by base-model perplexity (top-8192 → top-2048), but this step needs the base model — we defer to Phase 4b.

**Output**: `steering_data.json` per cluster (saved to HF repo qwen35-a3b-sae-phase2/).

**Budget**: ~5 min (no model loading, just tokenization + SAE forward + reconstruction).

## Cell 1 — Setup

In [ ]:
import sys, subprocess
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)
try:
    import safetensors, huggingface_hub
except Exception:
    pip('install', '-q', 'safetensors', 'huggingface_hub==1.5.0', 'hf_transfer', 'datasets', 'pandas')

import json, time, os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download, HfApi, login
from safetensors.numpy import load_file
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t: login(token=t); print('HF auth OK')
except: pass

OUT = Path('/content/phase4a'); OUT.mkdir(exist_ok=True)
HF_PHASE1 = 'caiovicentino1/qwen35-a3b-thinking-traces'
HF_PHASE2 = 'caiovicentino1/qwen35-a3b-sae-phase2'
N_LATENTS = 15
TOP_PER_CLUSTER = 8192  # paper uses 8192, later filtered to 2048 by perplexity in Phase 4b

## Cell 2 — Load Phase 1 + Phase 2 data

In [ ]:
# Phase 1: activations + sentences + prompt_idxs
p1_path = snapshot_download(HF_PHASE1, repo_type='dataset', cache_dir='/content/cache')
data = load_file(str(Path(p1_path) / 'activations.safetensors'))
activations = torch.from_numpy(data['activations'].astype(np.float32))
prompt_idxs = data['prompt_idxs']
sentences = json.load(open(Path(p1_path) / 'sentences.json'))
N, D = activations.shape
print(f'Phase 1: {N} sentences, d_model={D}')

# Phase 2: SAE checkpoint
p2_path = snapshot_download(HF_PHASE2, repo_type='model', cache_dir='/content/cache')
sae_state = torch.load(str(Path(p2_path) / f'sae_n{N_LATENTS}.pt'), map_location='cpu')
print(f'SAE keys: {list(sae_state.keys())}')

# Phase 3: cluster labels
with open(Path(p2_path) / f'cluster_labels_n{N_LATENTS}.json') as f:
    cluster_labels = json.load(f)
print(f'Labels: {len(cluster_labels)} clusters')

## Cell 3 — SAE forward → get top1 cluster + activation per sentence

In [ ]:
# Rebuild SAE and run forward
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k=3):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.zeros(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v

sae = TopKSAE(D, N_LATENTS, k=3)
sae.load_state_dict(sae_state)
sae.eval()

# Forward on all activations
with torch.no_grad():
    z, top_idx, top_val = sae.encode(activations)
    # top-1 per sentence
    top1_latent = top_idx[:, 0].numpy()
    top1_value = top_val[:, 0].numpy()
print(f'top1 distribution: {np.bincount(top1_latent)}')

## Cell 4 — Reconstruct prefixes: for each target sentence, get preceding sentences of same prompt

In [ ]:
# Group sentences by prompt_idx, preserve order
from collections import defaultdict
prompt_to_sent_indices = defaultdict(list)
for i, pid in enumerate(prompt_idxs):
    prompt_to_sent_indices[int(pid)].append(i)

# Load MMLU-Pro prompts (same 2000 sampled)
from huggingface_hub import snapshot_download
import pandas as pd
mmlu_path = snapshot_download('TIGER-Lab/MMLU-Pro', repo_type='dataset', cache_dir='/content/cache')
parquet_files = sorted(Path(mmlu_path).rglob('*.parquet'))
test_file = [f for f in parquet_files if 'test' in f.name.lower()][0]
mmlu_df = pd.read_parquet(test_file)
mmlu_items = mmlu_df.to_dict('records')
import random
random.seed(42)
sample_idx = random.sample(range(len(mmlu_items)), 2000)
prompts = [mmlu_items[i] for i in sample_idx]
print(f'MMLU prompts loaded: {len(prompts)}')

def format_prompt(ex):
    # Handle both list and numpy array options
    opts = list(ex['options']) if hasattr(ex['options'], '__len__') else ex['options']
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(opts))
    return (f"Task: Answer the question below. Explain your reasoning step by step.\n\n"
            f"Question: {ex['question']}\n\nOptions:\n{choices}\n\n"
            f"Step by step answer:")

## Cell 5 — Build per-cluster training data (top-8192 by SAE activation)

In [ ]:
steering_data = {}  # cluster_id -> list of {prompt, prefix, target, sae_activation}

for c in range(N_LATENTS):
    mask = top1_latent == c
    cluster_sent_idxs = np.where(mask)[0]
    if len(cluster_sent_idxs) == 0:
        steering_data[c] = []; continue
    # Sort by activation
    cluster_vals = top1_value[cluster_sent_idxs]
    sorted_idx = cluster_sent_idxs[np.argsort(-cluster_vals)][:TOP_PER_CLUSTER]
    entries = []
    for si in sorted_idx:
        pid = int(prompt_idxs[si])
        if pid >= len(prompts): continue
        # Reconstruct prefix: all sentences of same prompt with lower si (earlier in generation)
        same_prompt_sents = prompt_to_sent_indices[pid]
        same_prompt_sents_sorted = sorted(same_prompt_sents)
        try:
            pos_in_prompt = same_prompt_sents_sorted.index(si)
        except ValueError:
            continue
        prefix_sents = [sentences[same_prompt_sents_sorted[j]] for j in range(pos_in_prompt)]
        target_sent = sentences[si]
        entries.append(dict(
            prompt=format_prompt(prompts[pid]),
            prefix=' '.join(prefix_sents),
            target=target_sent,
            sae_activation=float(top1_value[si]),
            prompt_idx=pid,
            sent_idx=int(si)))
    steering_data[c] = entries
    print(f'Cluster {c} ({cluster_labels[str(c)]["title"]}): {len(entries)} entries')

# Save
with open(OUT / 'steering_data.json', 'w') as f:
    json.dump(steering_data, f, ensure_ascii=False)
print(f'\n✅ saved {sum(len(v) for v in steering_data.values())} total examples across {N_LATENTS} clusters')

## Cell 6 — Upload to HF

In [ ]:
import os
fn = OUT / 'steering_data.json'
size_mb = os.path.getsize(fn) / 1e6
print(f'Upload size: {size_mb:.1f} MB')

api = HfApi()
try:
    api.upload_file(path_or_fileobj=str(fn),
                    path_in_repo=f'steering_data_n{N_LATENTS}.json',
                    repo_id=HF_PHASE2, repo_type='model',
                    commit_message=f'Phase 4a: steering vector training data (top-{TOP_PER_CLUSTER}/cluster × {N_LATENTS} clusters)')
    print(f'✅ uploaded to https://huggingface.co/{HF_PHASE2}')
except Exception as e:
    print(f'upload error: {e}')

print(f'\nSample entry (cluster 3, {cluster_labels["3"]["title"]}):')
sample = steering_data[3][0] if steering_data[3] else None
if sample:
    print(f'  prompt (first 200 chars): {sample["prompt"][:200]}')
    print(f'  prefix (first 200 chars): {sample["prefix"][:200] or "<start of generation>"}')
    print(f'  target: {sample["target"][:200]}')
    print(f'  sae_activation: {sample["sae_activation"]:.3f}')